<a href="https://colab.research.google.com/github/digotill/AI-Coursework/blob/main/COA207_Route_Finding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

COA207 Portfolio - Component 2 (Implementation)
Student Name: Rodrigo Santiago Gentil

Student ID: F530392

Domain: Route Finding (Option 2)

Algorithm: A* Search

In [1]:
import heapq


# This is the graph of all space bases and how much it costs to travel between them
# I added a cost of 100 between Delta Mining and Epsilon Port for the Asteroid Storm twist
star_map = {
    'Prime Alpha': {'Beta Colony': 10, 'Gamma Outpost': 15},
    'Beta Colony': {'Prime Alpha': 10, 'Delta Mining': 10, 'Eta Research': 30},
    'Gamma Outpost': {'Prime Alpha': 15, 'Delta Mining': 12, 'Zeta Station': 15},
    'Delta Mining': {'Beta Colony': 10, 'Gamma Outpost': 12, 'Epsilon Port': 100}, # Storm here
    'Zeta Station': {'Gamma Outpost': 15, 'Epsilon Port': 10, 'Kappa Forge': 25},
    'Epsilon Port': {'Delta Mining': 100, 'Zeta Station': 10, 'Theta Relay': 15},
    'Eta Research': {'Beta Colony': 30, 'Theta Relay': 15},
    'Theta Relay': {'Epsilon Port': 15, 'Eta Research': 15, 'Iota Gateway': 10},
    'Kappa Forge': {'Zeta Station': 25, 'Lambda Point': 20},
    'Iota Gateway': {'Theta Relay': 10, 'Lambda Point': 15, 'Mu Array': 20},
    'Lambda Point': {'Kappa Forge': 20, 'Iota Gateway': 15, 'Nu Nexus': 10},
    'Mu Array': {'Iota Gateway': 20, 'Xi Vault': 15},
    'Nu Nexus': {'Lambda Point': 10, 'Omega Hub': 15},
    'Xi Vault': {'Mu Array': 15, 'Omega Hub': 10},
    'Omega Hub': {'Nu Nexus': 15, 'Xi Vault': 10}
}

# Heuristics (h): estimated straight-line distance to Omega Hub
# These are admissible since they don't overestimate the real distance
heuristics = {
    'Prime Alpha': 65, 'Beta Colony': 58, 'Gamma Outpost': 55,
    'Delta Mining': 50, 'Zeta Station': 45, 'Epsilon Port': 42,
    'Eta Research': 40, 'Theta Relay': 30, 'Kappa Forge': 35,
    'Iota Gateway': 25, 'Lambda Point': 18, 'Mu Array': 15,
    'Nu Nexus': 10, 'Xi Vault': 8, 'Omega Hub': 0
}


def a_star_search(graph, heuristics, start, goal, verbose=True):
    # frontier is a priority queue to keep track of nodes to visit
    # stores (f_score, node)
    frontier = []
    heapq.heappush(frontier, (heuristics[start], start))

    # came_from helps us backtrack to find the final path
    came_from = {start: None}

    # g_score is the actual cost found so far to get to a node
    g_score = {start: 0}

    if verbose:
        print(f"\n--- INITIATING A* SEARCH: {start} -> {goal} ---")

    while frontier:
        # Get the node with the lowest f_score (g + h)
        current_f, current_node = heapq.heappop(frontier)

        if verbose:
            print(f"\n[System] Expanding node: {current_node} (f-score: {current_f})")

        # Goal check
        if current_node == goal:
            if verbose:
                print(f"\n*** GOAL REACHED: {goal} ***")

            # Trace back to build the path list
            path = []
            while current_node is not None:
                path.append(current_node)
                current_node = came_from[current_node]
            path.reverse()
            return path, g_score[goal]

        # Check all connected bases
        for neighbor, edge_cost in graph[current_node].items():

            # Warn if the drone sees a storm
            if edge_cost >= 100 and verbose:
                print(f"  [WARNING] High-cost hazard detected to {neighbor}: Asteroid Storm! (Cost: {edge_cost})")

            # Calculate total cost to get to this neighbor
            tentative_g_score = g_score[current_node] + edge_cost

            # If this path is better than any we found before, save it
            if neighbor not in g_score or tentative_g_score < g_score[neighbor]:
                came_from[neighbor] = current_node
                g_score[neighbor] = tentative_g_score

                # f(n) = g(n) + h(n)
                f_score = tentative_g_score + heuristics[neighbor]
                heapq.heappush(frontier, (f_score, neighbor))

                if verbose:
                    print(f"  -> Valid path to {neighbor}. New g-score: {tentative_g_score}, f-score: {f_score}")

    return None, float('inf') # Fail case



def run_tests():
    print("======================================================")
    print("TEST 1: Normal Run (Avoiding the Storm)")
    print("======================================================")
    path, cost = a_star_search(star_map, heuristics, 'Prime Alpha', 'Omega Hub', verbose=True)
    print(f"\nFINAL PATH: {' -> '.join(path)}")
    print(f"TOTAL COST: {cost}")

    print("\n\n======================================================")
    print("TEST 2: Storm Dissipates (Direct Path)")
    print("======================================================")
    # Remove the storm to see if the AI adapts
    modified_map = {node: edges.copy() for node, edges in star_map.items()}
    modified_map['Delta Mining']['Epsilon Port'] = 10
    modified_map['Epsilon Port']['Delta Mining'] = 10

    path_clear, cost_clear = a_star_search(modified_map, heuristics, 'Prime Alpha', 'Omega Hub', verbose=False)
    print("[System] Storm cleared. Re-checking...")
    print(f"FINAL PATH: {' -> '.join(path_clear)}")
    print(f"TOTAL COST: {cost_clear}")

    print("\n\n======================================================")
    print("TEST 3: Goal Blocked (Quarantine)")
    print("======================================================")
    # Cut off Omega Hub to test error handling
    quarantine_map = {node: edges.copy() for node, edges in star_map.items()}
    quarantine_map['Nu Nexus'].pop('Omega Hub')
    quarantine_map['Xi Vault'].pop('Omega Hub')

    path_impossible, cost_impossible = a_star_search(quarantine_map, heuristics, 'Prime Alpha', 'Omega Hub', verbose=False)
    if path_impossible is None:
        print("[System] Error: Goal is unreachable. Path returned None.")

# Run everything
run_tests()

TEST 1: Normal Run (Avoiding the Storm)

--- INITIATING A* SEARCH: Prime Alpha -> Omega Hub ---

[System] Expanding node: Prime Alpha (f-score: 65)
  -> Valid path to Beta Colony. New g-score: 10, f-score: 68
  -> Valid path to Gamma Outpost. New g-score: 15, f-score: 70

[System] Expanding node: Beta Colony (f-score: 68)
  -> Valid path to Delta Mining. New g-score: 20, f-score: 70
  -> Valid path to Eta Research. New g-score: 40, f-score: 80

[System] Expanding node: Delta Mining (f-score: 70)
  [WARNING] High-cost hazard detected to Epsilon Port: Asteroid Storm! (Cost: 100)
  -> Valid path to Epsilon Port. New g-score: 120, f-score: 162

[System] Expanding node: Gamma Outpost (f-score: 70)
  -> Valid path to Zeta Station. New g-score: 30, f-score: 75

[System] Expanding node: Zeta Station (f-score: 75)
  -> Valid path to Epsilon Port. New g-score: 40, f-score: 82
  -> Valid path to Kappa Forge. New g-score: 55, f-score: 90

[System] Expanding node: Eta Research (f-score: 80)
  -> Va